# Semana 17 · ¿Qué velocidad hace falta para no caer?

Este cuaderno acompaña a la sesión de proyecto de la semana 17: velocidades orbitales, velocidad de escape y el cañón de Newton.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Velocidad orbital, de escape y periodo

Para una órbita circular de radio $r$ alrededor de la Tierra:

$$v_\text{orb} = \sqrt{\frac{GM}{r}}, \qquad v_\text{esc} = \sqrt{\frac{2GM}{r}}, \qquad T = 2\pi\sqrt{\frac{r^3}{GM}}.$$

Usamos directamente $GM_\text{T} = 3{,}986 \times 10^{14}$ m³/s², que se conoce mejor que $G$ y $M$ por separado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

GM = 3.986e14        # m³/s², Tierra
R = 6.371e6          # m, radio medio de la Tierra

def v_orbital(r):  return np.sqrt(GM / r)
def v_escape(r):   return np.sqrt(2 * GM / r)
def periodo(r):    return 2 * np.pi * np.sqrt(r**3 / GM)

ejemplos = {          # altura sobre la superficie, en km (aproximada)
    "a ras de suelo": 0,
    "Estación Espacial": 400,
    "GPS": 20_200,
    "geoestacionario": 35_786,
    "Luna": 384_400 - 6_371,
}
for nombre, h_km in ejemplos.items():
    r = R + h_km * 1e3
    print(f"{nombre:18s} r = {r/1e3:9.0f} km   v_orb = {v_orbital(r)/1e3:5.2f} km/s   v_esc = {v_escape(r)/1e3:5.2f} km/s   T = {periodo(r)/3600:7.2f} h")

Fíjate en tres cosas: el periodo de los satélites GPS es de casi 12 horas (medio día sidéreo); el de los geoestacionarios, casi 24 horas (un día sidéreo, 23 h 56 min); y el de la Luna sale unos 27,4 días, muy cerca de su periodo real, 27,3 días.

Ahora lo dibujamos en función de la altura, con escala logarítmica en el eje horizontal para que quepan de los 200 km a la Luna.

In [ ]:
h = np.logspace(5, np.log10(4e8), 300)       # alturas de 100 km a 400 000 km, en m
r = R + h

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.semilogx(h / 1e3, v_orbital(r) / 1e3, label="velocidad orbital")
ax1.semilogx(h / 1e3, v_escape(r) / 1e3, label="velocidad de escape")
ax1.set_xlabel("altura (km)"); ax1.set_ylabel("km/s"); ax1.legend(); ax1.grid(alpha=0.3, which="both")

ax2.loglog(h / 1e3, periodo(r) / 3600)
for nombre, h_km in list(ejemplos.items())[1:]:
    ax2.plot(h_km, periodo(R + h_km * 1e3) / 3600, "o")
    ax2.annotate(nombre, (h_km, periodo(R + h_km * 1e3) / 3600), textcoords="offset points", xytext=(-60, 8), fontsize=8)
ax2.set_xlabel("altura (km)"); ax2.set_ylabel("periodo (horas)"); ax2.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

**Ejercicio.** Busca, probando alturas o despejando, a qué altura el periodo es exactamente un día sidéreo (86 164 s). Comprueba que sale la altura geoestacionaria.

In [ ]:
# Tu código aquí

In [ ]:
# Solución: despejamos r de T = 2π·sqrt(r³/GM)
T_sid = 86164.0
r_geo = (GM * T_sid**2 / (4 * np.pi**2)) ** (1 / 3)
print(f"radio = {r_geo/1e3:.0f} km   altura = {(r_geo - R)/1e3:.0f} km")

## 2. El cañón de Newton

Simulamos una bala disparada en horizontal desde la superficie de la Tierra (sin aire, y con una montaña imaginaria que no estorba). La única fuerza es la gravedad, que apunta hacia el centro de la Tierra:

$$\vec{a} = -\frac{GM}{r^3}\,\vec{r}.$$

Usamos el método de **Verlet de velocidad** (o *leapfrog*), primo del de Euler-Cromer de la semana 12. En cada paso: medio empujón a la velocidad, un paso completo de posición y otro medio empujón con la aceleración nueva. Conserva la energía mucho mejor que Euler, y es el que usan los astrónomos para simular órbitas durante millones de años.

In [ ]:
def aceleracion(pos):
    r = np.linalg.norm(pos)
    return -GM / r**3 * pos

def canon(v0, t_max=13 * 3600, dt=2.0):
    """Dispara en horizontal desde (0, R) con velocidad v0 (m/s). Devuelve la trayectoria."""
    pos = np.array([0.0, R])
    vel = np.array([v0, 0.0])
    a = aceleracion(pos)
    tray = [pos.copy()]
    t = 0.0
    while t < t_max:
        vel += 0.5 * dt * a
        pos += dt * vel
        a = aceleracion(pos)
        vel += 0.5 * dt * a
        t += dt
        tray.append(pos.copy())
        if np.linalg.norm(pos) < R - 1:      # ha chocado con el suelo
            break
    return np.array(tray)

fig, ax = plt.subplots(figsize=(7, 7))
ax.add_patch(plt.Circle((0, 0), R / 1e3, color="lightblue"))
for v0 in [4000, 6000, 7905, 9000, 10500, 11200, 12000]:
    tr = canon(v0)
    ax.plot(tr[:, 0] / 1e3, tr[:, 1] / 1e3, label=f"{v0/1e3:.1f} km/s", lw=1)
ax.set_aspect("equal"); ax.set_xlim(-30000, 40000); ax.set_ylim(-50000, 12000)
ax.legend(loc="lower right", fontsize=8); ax.set_xlabel("km"); ax.set_ylabel("km")
ax.set_title("El cañón de Newton: misma dirección, distintas velocidades")
plt.show()

Las balas lentas caen; la de 7,9 km/s describe una circunferencia; las de 9 y 10,5 km/s, elipses cada vez más alargadas con el perigeo en el cañón; y las de 11,2 y 12 km/s se alejan sin volver (la simulación se corta a las 13 horas, cuando la elipse de 10,5 km/s ya ha dado una vuelta completa).

## 3. La energía decide

La energía total por unidad de masa es $\varepsilon = \tfrac{1}{2}v^2 - GM/r$. Debe mantenerse constante durante el vuelo. Si es negativa, la bala está ligada a la Tierra; si es positiva o cero, escapa.

**Ejercicio.** Calcula $\varepsilon$ al principio y al final de cada trayectoria del cañón (con la velocidad de salida y la posición inicial basta para el principio; para el final, necesitas que la función `canon` devuelva también la velocidad). Comprueba que no cambia y que su signo coincide con lo que ves en la gráfica.

In [ ]:
# Tu código aquí

In [ ]:
# Solución: energía inicial (se conserva, así que basta con calcularla al salir)
for v0 in [4000, 6000, 7905, 9000, 10500, 11200, 12000]:
    eps = 0.5 * v0**2 - GM / R
    estado = "ligada" if eps < 0 else "escapa"
    print(f"v0 = {v0/1e3:5.2f} km/s   energía por kg = {eps/1e6:7.2f} MJ/kg   → {estado}")
print(f"\nVelocidad de escape desde la superficie: {v_escape(R)/1e3:.3f} km/s")

Para comprobar que el método conserva la energía, modificamos un poco la simulación para que guarde la energía en cada paso, y la dibujamos para la órbita de 9 km/s durante varias vueltas.

In [ ]:
def energia_en_el_tiempo(v0, t_max=12 * 3600, dt=1.0):
    pos = np.array([0.0, R]); vel = np.array([v0, 0.0]); a = aceleracion(pos)
    E = []
    for _ in range(int(t_max / dt)):
        vel += 0.5 * dt * a; pos += dt * vel; a = aceleracion(pos); vel += 0.5 * dt * a
        E.append(0.5 * vel @ vel - GM / np.linalg.norm(pos))
    return np.array(E)

E = energia_en_el_tiempo(9000)
plt.plot(np.arange(len(E)) / 3600, (E - E[0]) / abs(E[0]))
plt.xlabel("tiempo (h)"); plt.ylabel("error relativo de la energía")
plt.title("Verlet: la energía oscila un poco, pero no se va"); plt.grid(alpha=0.3)
plt.show()

## 4. Experimenta

1. Cambia la Tierra por la Luna ($GM = 4{,}90 \times 10^{12}$ m³/s², $R = 1737$ km) y busca la velocidad de la órbita circular a ras de suelo y la de escape.
2. Dispara con 9 km/s pero con un ángulo de 30° hacia arriba sobre la horizontal. ¿Sigue entrando en órbita? Pista: una órbita cerrada vuelve a pasar por el punto de partida, y ahí está el suelo.
3. Cambia el método por el de Euler simple (actualiza la posición con la velocidad vieja y la velocidad con la aceleración vieja). ¿Qué le pasa a la órbita de 7,9 km/s al cabo de varias vueltas?

In [ ]:
# Tu experimento aquí